# Facade-Renderer: Pix2Pix Conditional GAN (cGAN) on Kaggle

This notebook trains a **Pix2Pix Conditional GAN** to translate architectural blueprints and semantic label maps into photorealistic building facades using the **CMP Facades dataset** (`vikramtiwari/pix2pix-dataset`).

### Features:
1. **Auto-Hardware Detection:** Automatically enables CUDA GPU acceleration on Kaggle (P100 / T4).
2. **Dataset Flexibility:** Automatically detects `/kaggle/input/pix2pix-dataset`, `kagglehub`, or Berkeley direct fallback.
3. **U-Net 256 Architecture:** Downsampling-upsampling with skip connections preserving spatial alignment.
4. **70x70 PatchGAN:** High-frequency texture and boundary discriminator.
5. **Combined Loss:** Adversarial Loss + $\lambda \times L_1$ Loss ($\lambda = 100$).
6. **Inline Visual Monitoring:** Displays generated building facades every few epochs directly in the notebook.
7. **Downloadable Outputs:** Saves model checkpoints to `/kaggle/working/checkpoints/`.

## 1. Setup Environment & Device Configuration

In [ ]:
import os
import glob
import time
import math
import random
import tarfile
import urllib.request
from typing import Tuple, Optional
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

# Device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Active device:   {device}')
if device.type == 'cuda':
    print(f'GPU Name:        {torch.cuda.get_device_name(0)}')
    print(f'Available VRAM:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('Note: Running on CPU. For faster training on Kaggle, turn on GPU in the notebook settings (Notebook options -> Accelerator -> GPU T4 or P100).')

## 2. Locate or Download Dataset
The notebook checks:
1. Attached Kaggle dataset `/kaggle/input/pix2pix-dataset`
2. Local directory `D:\Datasets\facades` or `./facades`
3. Downloads via `kagglehub` or official repository.

In [ ]:
def setup_dataset():
    # Potential paths on Kaggle and local machines
    candidates = [
        '/kaggle/input/pix2pix-dataset/facades',
        '/kaggle/input/pix2pix-dataset',
        r'D:\Datasets\facades',
        r'D:\Datasets\datasets\vikramtiwari\pix2pix-dataset\facades',
        './facades',
    ]
    for path in candidates:
        if os.path.exists(path) and (os.path.exists(os.path.join(path, 'train')) or os.path.exists(os.path.join(path, 'facades', 'train'))):
            actual_path = os.path.join(path, 'facades') if os.path.exists(os.path.join(path, 'facades', 'train')) else path
            print(f'Found existing dataset at: {actual_path}')
            return actual_path

    # If not found, download CMP Facades
    dest_dir = '/kaggle/working/facades' if os.path.exists('/kaggle') else './facades'
    if not os.path.exists(dest_dir):
        print('Downloading CMP Facades dataset archive (~30 MB)...')
        os.makedirs(dest_dir, exist_ok=True)
        url = 'http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz'
        tar_path = os.path.join(dest_dir, 'facades.tar.gz')
        urllib.request.urlretrieve(url, tar_path)
        parent_dir = os.path.dirname(dest_dir)
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=parent_dir)
        if os.path.exists(tar_path):
            os.remove(tar_path)
        print(f'Extracted dataset to: {dest_dir}')
    return dest_dir

DATASET_DIR = setup_dataset()
print(f'Using dataset directory: {DATASET_DIR}')
train_files = glob.glob(os.path.join(DATASET_DIR, 'train', '*.jpg'))
val_files = glob.glob(os.path.join(DATASET_DIR, 'val', '*.jpg'))
test_files = glob.glob(os.path.join(DATASET_DIR, 'test', '*.jpg'))
print(f'Samples -> Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}')

## 3. Dataset & DataLoaders with Slicing & Augmentations
CMP Facades paired samples store building photos (left) and blueprints (right) in a single $512 \times 256$ image.
- We slice each image horizontally.
- **Direction:** `BtoA` (blueprint $\to$ photo).
- Normalizes pixels to $[-1, 1]$ matching Generator `Tanh` output.

In [ ]:
class CMPFacadesDataset(Dataset):
    def __init__(self, root_dir: str, split: str = 'train', direction: str = 'BtoA', img_size: int = 256):
        super().__init__()
        self.split = split
        self.direction = direction.upper()
        self.img_size = img_size
        
        split_dir = os.path.join(root_dir, split)
        target_dir = split_dir if os.path.exists(split_dir) else root_dir
        self.image_paths = sorted(
            glob.glob(os.path.join(target_dir, '*.jpg')) + glob.glob(os.path.join(target_dir, '*.png'))
        )
        if not self.image_paths:
            raise FileNotFoundError(f'No images found in {target_dir}')

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        composite_img = Image.open(self.image_paths[idx]).convert('RGB')
        w, h = composite_img.size
        half_w = w // 2
        left = composite_img.crop((0, 0, half_w, h))
        right = composite_img.crop((half_w, 0, w, h))

        condition = right if self.direction == 'BTOA' else left
        target = left if self.direction == 'BTOA' else right

        if self.split == 'train':
            jitter_size = int(self.img_size * 286 / 256)
            condition = TF.resize(condition, [jitter_size, jitter_size], interpolation=T.InterpolationMode.BICUBIC)
            target = TF.resize(target, [jitter_size, jitter_size], interpolation=T.InterpolationMode.BICUBIC)
            i, j, h_crop, w_crop = T.RandomCrop.get_params(condition, output_size=(self.img_size, self.img_size))
            condition = TF.crop(condition, i, j, h_crop, w_crop)
            target = TF.crop(target, i, j, h_crop, w_crop)
            if random.random() > 0.5:
                condition = TF.hflip(condition)
                target = TF.hflip(target)
        else:
            condition = TF.resize(condition, [self.img_size, self.img_size], interpolation=T.InterpolationMode.BICUBIC)
            target = TF.resize(target, [self.img_size, self.img_size], interpolation=T.InterpolationMode.BICUBIC)

        normalize = T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        return normalize(TF.to_tensor(condition)), normalize(TF.to_tensor(target))

# Build DataLoaders
BATCH_SIZE = 4 if torch.cuda.is_available() else 1
train_dataset = CMPFacadesDataset(DATASET_DIR, split='train')
val_dataset = CMPFacadesDataset(DATASET_DIR, split='val')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
print(f'Batch size: {BATCH_SIZE} | Train batches per epoch: {len(train_loader)}')

### Visualize a Training Sample Pair

In [ ]:
def denorm(t):
    return (t.clamp(-1, 1) + 1.0) / 2.0

cond_sample, target_sample = train_dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(denorm(cond_sample).permute(1, 2, 0).numpy())
axes[0].set_title('Condition (Blueprint / Mask)')
axes[0].axis('off')

axes[1].imshow(denorm(target_sample).permute(1, 2, 0).numpy())
axes[1].set_title('Target (Real Building Photo)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Models: U-Net Generator & PatchGAN Discriminator

In [ ]:
def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif classname.find('BatchNorm2d') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

class UNetDown(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, normalize: bool = True):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=False))
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.ReLU(inplace=False),
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        ]
        if dropout > 0.0:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x, skip_input):
        x = self.model(x)
        return torch.cat((x, skip_input), dim=1)

class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.down1 = UNetDown(in_channels, 64, normalize=False)  # 256 -> 128
        self.down2 = UNetDown(64, 128)                           # 128 -> 64
        self.down3 = UNetDown(128, 256)                          # 64 -> 32
        self.down4 = UNetDown(256, 512)                          # 32 -> 16
        self.down5 = UNetDown(512, 512)                          # 16 -> 8
        self.down6 = UNetDown(512, 512)                          # 8 -> 4
        self.down7 = UNetDown(512, 512)                          # 4 -> 2
        self.down8 = nn.Sequential(                              # 2 -> 1
            nn.LeakyReLU(0.2, inplace=False),
            nn.Conv2d(512, 512, 4, 2, 1, bias=False),
        )
        self.up1 = UNetUp(512, 512, dropout=0.5)                 # 1 -> 2
        self.up2 = UNetUp(1024, 512, dropout=0.5)                # 2 -> 4
        self.up3 = UNetUp(1024, 512, dropout=0.5)                # 4 -> 8
        self.up4 = UNetUp(1024, 512)                             # 8 -> 16
        self.up5 = UNetUp(1024, 256)                             # 16 -> 32
        self.up6 = UNetUp(512, 128)                              # 32 -> 64
        self.up7 = UNetUp(256, 64)                               # 64 -> 128
        self.final = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.ConvTranspose2d(128, out_channels, 4, 2, 1),
            nn.Tanh(),
        )
    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)
        u1 = self.up1(d8, d7)
        u2 = self.up2(u1, d6)
        u3 = self.up3(u2, d5)
        u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3)
        u6 = self.up6(u5, d2)
        u7 = self.up7(u6, d1)
        return self.final(u7)

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()
        def block(in_f, out_f, stride=2, normalize=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride=stride, padding=1, bias=not normalize)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=False))
            return layers
        self.model = nn.Sequential(
            *block(in_channels, 64, stride=2, normalize=False),
            *block(64, 128, stride=2),
            *block(128, 256, stride=2),
            *block(256, 512, stride=1),
            nn.Conv2d(512, 1, 4, stride=1, padding=1),
        )
    def forward(self, condition, target):
        return self.model(torch.cat((condition, target), dim=1))

# Instantiate models and apply weight initialization
generator = UNetGenerator().to(device)
discriminator = PatchGANDiscriminator().to(device)
generator.apply(weights_init_normal)
discriminator.apply(weights_init_normal)
print('Networks initialized with Normal(0.0, 0.02) weights.')

## 5. Loss Functions & Optimizers
- Adversarial Loss: `BCEWithLogitsLoss`
- Structural Reconstruction Loss: `L1Loss`
- Combined Generator Objective: $\mathcal{L}_G = \mathcal{L}_{GAN} + 100 \times \mathcal{L}_{L1}$

In [ ]:
criterion_GAN = nn.BCEWithLogitsLoss().to(device)
criterion_L1 = nn.L1Loss().to(device)
LAMBDA_L1 = 100.0
LR = 0.0002
BETA1 = 0.5

optimizer_G = torch.optim.Adam(generator.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(BETA1, 0.999))

# Checkpoint directory
CHECKPOINT_DIR = '/kaggle/working/checkpoints' if os.path.exists('/kaggle') else './outputs/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

## 6. Visual Sampling Callback

In [ ]:
def plot_sample_comparison(epoch, num_samples=3):
    generator.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(10, num_samples * 3.3))
    if num_samples == 1:
        axes = [axes]
    
    with torch.no_grad():
        for i, (condition, real_target) in enumerate(val_loader):
            if i >= num_samples:
                break
            condition = condition.to(device)
            fake_target = generator(condition)
            
            cond_img = denorm(condition[0].cpu()).permute(1, 2, 0).numpy()
            real_img = denorm(real_target[0]).permute(1, 2, 0).numpy()
            fake_img = denorm(fake_target[0].cpu()).permute(1, 2, 0).numpy()
            
            axes[i][0].imshow(cond_img)
            axes[i][0].set_title(f'Condition #{i+1}')
            axes[i][0].axis('off')
            
            axes[i][1].imshow(real_img)
            axes[i][1].set_title('Real Ground Truth')
            axes[i][1].axis('off')
            
            axes[i][2].imshow(fake_img)
            axes[i][2].set_title(f'Generated (Epoch {epoch})')
            axes[i][2].axis('off')
            
    plt.suptitle(f'Visual Results at Epoch {epoch}', fontsize=14)
    plt.tight_layout()
    plt.show()
    generator.train()

## 7. Training Loop
Configured for 200 epochs by default. Feel free to adjust `EPOCHS` (e.g. 50-100 epochs on Kaggle P100 takes only ~15-25 minutes).

In [ ]:
EPOCHS = 100  # Adjust as needed (200 is full paper training)
SAMPLE_INTERVAL = 10
CHECKPOINT_INTERVAL = 20

history_g_loss = []
history_d_loss = []
history_l1_loss = []

print('Starting training...')
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_d = 0.0
    epoch_g = 0.0
    epoch_l1 = 0.0
    generator.train()
    discriminator.train()

    for i, (condition, real_target) in enumerate(train_loader):
        condition = condition.to(device)
        real_target = real_target.to(device)

        # 1) Train Generator: minimize adversarial loss + L1 loss
        optimizer_G.zero_grad()
        fake_target = generator(condition)
        pred_fake = discriminator(condition, fake_target)
        valid = torch.ones_like(pred_fake, device=device)
        fake_label = torch.zeros_like(pred_fake, device=device)

        loss_G_adv = criterion_GAN(pred_fake, valid)
        loss_G_l1 = criterion_L1(fake_target, real_target)
        loss_G = loss_G_adv + (LAMBDA_L1 * loss_G_l1)
        loss_G.backward()
        optimizer_G.step()

        # 2) Train Discriminator: real pairs vs fake pairs
        optimizer_D.zero_grad()
        pred_real = discriminator(condition, real_target)
        loss_D_real = criterion_GAN(pred_real, valid)
        pred_fake_d = discriminator(condition, fake_target.detach())
        loss_D_fake = criterion_GAN(pred_fake_d, fake_label)
        loss_D = 0.5 * (loss_D_real + loss_D_fake)
        loss_D.backward()
        optimizer_D.step()

        epoch_d += loss_D.item()
        epoch_g += loss_G.item()
        epoch_l1 += loss_G_l1.item()

    avg_d = epoch_d / len(train_loader)
    avg_g = epoch_g / len(train_loader)
    avg_l1 = epoch_l1 / len(train_loader)
    history_d_loss.append(avg_d)
    history_g_loss.append(avg_g)
    history_l1_loss.append(avg_l1)

    elapsed = time.time() - start_time
    print(f'Epoch [{epoch:03d}/{EPOCHS:03d}] | D Loss: {avg_d:.4f} | G Loss: {avg_g:.4f} | L1: {avg_l1:.4f} | Time: {elapsed/60:.1f}m')

    # Visual progress
    if epoch % SAMPLE_INTERVAL == 0 or epoch == 1:
        plot_sample_comparison(epoch)

    # Save checkpoints
    if epoch % CHECKPOINT_INTERVAL == 0 or epoch == EPOCHS:
        torch.save(generator.state_dict(), os.path.join(CHECKPOINT_DIR, f'generator_epoch_{epoch:03d}.pth'))
        torch.save(generator.state_dict(), os.path.join(CHECKPOINT_DIR, 'generator_latest.pth'))
        print(f'Saved generator checkpoint at epoch {epoch}')

print('Training complete! Final model saved to generator_latest.pth')

## 8. Loss Curves Plotting

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_g_loss, label='Generator Loss')
plt.plot(history_d_loss, label='Discriminator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Adversarial Training Curves')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_l1_loss, label='L1 Reconstruction Loss', color='green')
plt.xlabel('Epoch')
plt.ylabel('L1 MAE')
plt.title('Pixelwise L1 Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Evaluation & Inference on Test Set

In [ ]:
test_dataset = CMPFacadesDataset(DATASET_DIR, split='test')
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

generator.eval()
total_l1 = 0.0
total_psnr = 0.0

with torch.no_grad():
    for i, (condition, real_target) in enumerate(test_loader):
        condition = condition.to(device)
        real_target = real_target.to(device)
        fake_target = generator(condition)
        
        cond_d = denorm(condition)
        real_d = denorm(real_target)
        fake_d = denorm(fake_target)
        
        l1 = torch.mean(torch.abs(fake_d - real_d)).item()
        mse = torch.mean((fake_d - real_d)**2).item()
        psnr = 20 * math.log10(1.0 / math.sqrt(mse)) if mse > 0 else float('inf')
        
        total_l1 += l1
        total_psnr += psnr

print(f'Test Samples: {len(test_loader)}')
print(f'Mean L1 Error: {total_l1 / len(test_loader):.4f}')
print(f'Mean PSNR:     {total_psnr / len(test_loader):.2f} dB')